# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binteaamer/FlyrankInternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window


**One row =** One content_hash_id (a single webpage) aggregated over a fixed 30-day window

**Which table(s)?**
- `fact_content_daily_performance` (daily GSC + GA4 metrics, partitioned by month)

**Time window:**
- Feature window: days [end_date - 60 to end_date - 30] (the 30 days BEFORE prediction)
- Label window: days [end_date - 30 to end_date] (the 30 days WE PREDICT)
- end_date = MAX(report_date) from the monthly partition (e.g., 2026-03-31)
- Example: Using month=2026-03, we see which pages improved in late March

**Label (what we predict):**
- `is_improving` = 1 if BOTH:
  - Position improved: avg_position_last30 < avg_position_prev30 - 1.0 (ranked ≥1 position higher)
  - AND impressions grew: impressions_last30 >= 1.1 * impressions_prev30 (≥10% growth)
- Else 0

**One thing we deliberately exclude:**
- `gsc_impressions`, `gsc_clicks` from the label window (last_30) itself
  - Why? They're used to calculate the label; using them as features = leakage

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

### FEATURES (what the model learns from)
- `impressions_prev30` — Total impressions, days [end-60 to end-30]
- `clicks_prev30` — Total clicks, days [end-60 to end-30]
- `avg_position_prev30` — Average position (lower = better), days [end-60 to end-30]
- `visible_queries` — How many distinct queries this page ranks for (90-day window)
- `ctr_prev30` — Click-through rate, days [end-60 to end-30]

### LABEL (what we predict)
- `is_improving` — 1 if position improved ≥1 rank AND impressions grew ≥10%, else 0
  - Calculated only from non-overlapping windows (prev30 vs last30)

### CONTEXT (grouping/joins only, NOT features)
- `client_hash_id` — Pseudonymous client identifier
- `content_hash_id` — Pseudonymous content identifier
- `report_date` — End date of measurement window

### EXCLUDED (and why)
- `impressions_last30`, `clicks_last30` from last_30 window → LEAKAGE (used in label calculation)
- `gsc_avg_position` from last_30 window → LEAKAGE (used in label calculation)
- `trend_direction`, `trend_pct` → Would encode the label already
- `days_since_update`, `update_recency` if from label window → Could leak freshness

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# Setup: Connect to warehouse
import duckdb
import pandas as pd
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("✅ Connected to Hugging Face warehouse")

✅ Connected to Hugging Face warehouse


In [2]:
# QUERY 1: Grain check
# Claim: One row = one content_hash_id per month, not per day

grain_check = con.sql(f"""
    WITH daily_data AS (
        SELECT content_hash_id, report_date, gsc_impressions
        FROM {TABLES['fact_daily']}
    ),
    aggregated AS (
        SELECT
            content_hash_id,
            COUNT(DISTINCT report_date) AS days_of_data,
            COUNT(*) AS rows_in_data
        FROM daily_data
        GROUP BY content_hash_id
    )
    SELECT
        COUNT(*) AS unique_content_items,
        AVG(days_of_data) AS avg_days_per_content,
        MIN(days_of_data) AS min_days,
        MAX(days_of_data) AS max_days
    FROM aggregated
""").df()

print("✅ QUERY 1: Grain Verification")
print(grain_check)
print(f"\nInterpretation: Each content_hash_id has ~{grain_check['avg_days_per_content'].values[0]:.0f} days of data in March 2026.")
print("After aggregation, one row will = one content_hash_id for March.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ QUERY 1: Grain Verification
   unique_content_items  avg_days_per_content  min_days  max_days
0                331437             29.693058         1        31

Interpretation: Each content_hash_id has ~30 days of data in March 2026.
After aggregation, one row will = one content_hash_id for March.


In [3]:
# QUERY 2: Slice size and date coverage
# Claim: Our slice has X rows across [start_date to end_date]

slice_facts = con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(DISTINCT report_date) AS days_covered,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(*) AS total_rows,
        ROUND(COUNT(*) / COUNT(DISTINCT content_hash_id), 1) AS avg_rows_per_content
    FROM {TABLES['fact_daily']}
""").df()

print("✅ QUERY 2: Slice Size & Coverage")
print(slice_facts)
print(f"\nInterpretation: March 2026 has {slice_facts['days_covered'].values[0]} days of data,")
print(f"covering {slice_facts['unique_content_items'].values[0]} unique content items.")

✅ QUERY 2: Slice Size & Coverage
  start_date   end_date  days_covered  unique_content_items  total_rows  \
0 2026-03-01 2026-03-31            31                331437     9841378   

   avg_rows_per_content  
0                  29.7  

Interpretation: March 2026 has 31 days of data,
covering 331437 unique content items.


In [4]:
# QUERY 3: Data availability - how many rows have the fields we need?
# Claim: When we filter on data quality flags, how much data survives?

availability = con.sql(f"""
    WITH raw_data AS (
        SELECT
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_data_available
        FROM {TABLES['fact_daily']}
    )
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_impressions IS NOT NULL THEN 1 ELSE 0 END) AS has_impressions,
        SUM(CASE WHEN gsc_avg_position IS NOT NULL THEN 1 ELSE 0 END) AS has_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_true,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ga4_available
    FROM raw_data
""").df()

print("✅ QUERY 3: Data Availability (IS TRUE filter)")
print(availability)
print(f"\nInterpretation: {availability['pct_ga4_available'].values[0]}% of rows have GA4 data available.")
print("We'll filter on ga4_data_available IS TRUE for quality.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ QUERY 3: Data Availability (IS TRUE filter)
   total_rows  has_impressions  has_position  ga4_available_true  \
0     9841378        9841378.0     3611061.0            413966.0   

   pct_ga4_available  
0                4.2  

Interpretation: 4.2% of rows have GA4 data available.
We'll filter on ga4_data_available IS TRUE for quality.


In [6]:
# FEATURE BUILDING: Aggregate daily data into feature vectors
# Working month: 2026-03

features_query = f"""
    WITH bounds AS (
        -- Find the end date of this month
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.content_hash_id,
            f.client_hash_id,

            -- FEATURE 1: Previous 30-day impressions
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS impressions_prev30,

            -- FEATURE 2: Previous 30-day clicks
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_clicks ELSE 0 END) AS clicks_prev30,

            -- FEATURE 3: Previous 30-day average position
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_avg_position ELSE NULL END) AS avg_position_prev30,

            -- FEATURE 4: Last 30-day average position (for label)
            AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                     THEN f.gsc_avg_position ELSE NULL END) AS avg_position_last30,

            -- FEATURE 5: Last 30-day impressions (for label)
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS impressions_last30

        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.ga4_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
    ),
    with_queries AS (
        SELECT
            w.content_hash_id,
            w.client_hash_id,
            w.impressions_prev30,
            w.clicks_prev30,
            w.avg_position_prev30,
            w.impressions_last30,
            w.avg_position_last30,

            -- FEATURE 5: Query diversity - COLUMN NAME CORRECTED
            COALESCE(q.content_visible_query_count, 0) AS visible_queries
        FROM windowed w
        LEFT JOIN {TABLES['fact_query_90d']} q
            ON w.content_hash_id = q.content_hash_id
    )
    SELECT * FROM with_queries
"""

features_df = con.sql(features_query).df()

print(f"✅ Built features for {len(features_df)} content items")
print("\nFeature DataFrame (first 10 rows):")
print(features_df.head(10))
print(f"\nFeature shape: {features_df.shape}")
print(f"\nData types:\n{features_df.dtypes}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Built features for 1493871 content items

Feature DataFrame (first 10 rows):
            content_hash_id           client_hash_id  impressions_prev30  \
0  content_810cf06597918291  client_9958f0a7ae1df715                11.0   
1  content_eb0aeedbcfaf2712  client_9958f0a7ae1df715                49.0   
2  content_651b8ba180f9beff  client_9958f0a7ae1df715                 5.0   
3  content_1f39e904c7351258  client_9958f0a7ae1df715                28.0   
4  content_a46986d3796592f9  client_9958f0a7ae1df715                89.0   
5  content_f5e11209b398d173  client_9958f0a7ae1df715                 0.0   
6  content_237e63fc00c8c7ad  client_9958f0a7ae1df715               642.0   
7  content_1021a22410889b2a  client_9958f0a7ae1df715                11.0   
8  content_f216815b76a1bdb8  client_9958f0a7ae1df715                 2.0   
9  content_6f4cc70af7be346e  client_9958f0a7ae1df715                17.0   

   clicks_prev30  avg_position_prev30  impressions_last30  \
0            0.0       

## Five Features (Available When?)

1. **impressions_prev30**
   - Knowable at decision moment because: Counted from GSC data already in the system,
     covering the 30 days BEFORE we make the prediction. No future data needed.

2. **clicks_prev30**
   - Knowable at decision moment because: Counted from GSC data already in the system,
     covering the 30 days BEFORE we make the prediction. No future data needed.

3. **avg_position_prev30**
   - Knowable at decision moment because: Averaged from daily position data in GSC,
     covering the 30 days BEFORE we make the prediction. No future data needed.

4. **visible_queries**
   - Knowable at decision moment because: Counted from the 90-day query-level fact table,
     which is refreshed as a snapshot. Tells us how many distinct queries this page ranks for.

5. **CTR (clicks / impressions)**
   - Knowable at decision moment because: Calculated from impressions_prev30 and clicks_prev30,
     both historical. CTR = clicks_prev30 / impressions_prev30 where impressions_prev30 > 0.

In [8]:
# THE TRAP: Add a leaky column on purpose, watch performance jump

# Add the label
features_df['is_improving'] = (
    (features_df['avg_position_last30'] < features_df['avg_position_prev30'] - 1.0) &
    (features_df['impressions_last30'] >= 1.1 * features_df['impressions_prev30'])
).astype(int)

print(f"Label distribution:\n{features_df['is_improving'].value_counts()}")

# NOW ADD THE TRAP: Use label-window data as a feature
features_df['position_change_leaky'] = (
    features_df['avg_position_prev30'] - features_df['avg_position_last30']
)

# Train two models: one with leakage, one without
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# LEAKY MODEL (with position_change_leaky)
X_leaky = features_df[['impressions_prev30', 'clicks_prev30', 'avg_position_prev30',
                        'visible_queries', 'position_change_leaky']].dropna()
y = features_df.loc[X_leaky.index, 'is_improving']

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.3, random_state=42, stratify=y
)
model_leaky = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, model_leaky.predict_proba(X_test_l)[:, 1])

print("\n🚨 LEAKY MODEL (using position change from LABEL window):")
print(f"   ROC-AUC: {leaky_auc:.3f}  ← Suspiciously high!")
print(classification_report(y_test_l, model_leaky.predict(X_test_l)))

# HONEST MODEL (without leaky column)
X_honest = features_df[['impressions_prev30', 'clicks_prev30', 'avg_position_prev30',
                         'visible_queries']].dropna()
y = features_df.loc[X_honest.index, 'is_improving']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_honest, y, test_size=0.3, random_state=42, stratify=y
)
model_honest = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_h, y_train_h)
honest_auc = roc_auc_score(y_test_h, model_honest.predict_proba(X_test_h)[:, 1])

print("\n✅ HONEST MODEL (using only pre-decision features):")
print(f"   ROC-AUC: {honest_auc:.3f}  ← Real performance")
print(classification_report(y_test_h, model_honest.predict(X_test_h)))

print(f"\n⚠️  LEAKAGE LESSON:")
print(f"   Leaky model: {leaky_auc:.3f}")
print(f"   Honest model: {honest_auc:.3f}")
print(f"   Difference: {leaky_auc - honest_auc:.3f} (this gap is the leakage)")

Label distribution:
is_improving
0    1434414
1      59457
Name: count, dtype: int64

🚨 LEAKY MODEL (using position change from LABEL window):
   ROC-AUC: 1.000  ← Suspiciously high!
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42829
           1       1.00      1.00      1.00     17837

    accuracy                           1.00     60666
   macro avg       1.00      1.00      1.00     60666
weighted avg       1.00      1.00      1.00     60666


✅ HONEST MODEL (using only pre-decision features):
   ROC-AUC: 1.000  ← Real performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     43227
           1       1.00      1.00      1.00     17837

    accuracy                           1.00     61064
   macro avg       1.00      1.00      1.00     61064
weighted avg       1.00      1.00      1.00     61064


⚠️  LEAKAGE LESSON:
   Leaky model: 1.000
   Honest model: 1.000
   Diff

## 4. Data Limits: What This Data Can NEVER Tell You

1. **Unbalanced Panel by Client**
   - Different clients have different history lengths (3–17 months in full warehouse)
   - Some clients have no GA4 data early on
   - Can't compare raw metrics across clients; must normalize

2. **Query Table is a Snapshot**
   - `fact_content_query_90d` is a fixed 90-day window (most recent ~3 months)
   - Predicting labels in that same window introduces leakage
   - Must treat June 2026 (final month) as a sealed test set

3. **Position Movements are Noisy**
   - Position varies day-to-day (SERP volatility, time of day, location)
   - Average smooths this but can't separate real rank improvement from noise
   - Should pair with impression growth to confirm (hence Option B3)

4. **Causation ≠ Correlation**
   - If a page ranks better, we don't know WHY (algorithm shift, our optimization,
     search volume increase)
   - This model predicts WHAT happens, not WHY it happens

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.